# LeetCode 1732 — Find the Highest Altitude bằng Python

Link đề: https://leetcode.com/problems/find-the-highest-altitude/

Độ khó: **Easy** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def largestAltitude(self, gain: list[int]) -> int:

```

Mục tiêu học:

1. Hiểu khái niệm **tổng tiền tố** (prefix sum): độ cao tại điểm `i` = tổng mọi `gain` đứng trước nó.
2. Phân biệt **mảng chênh lệch** (`gain`) với **mảng giá trị thật** (`altitudes`) — một ý tưởng gặp rất nhiều trong DSA.
3. Viết được mẫu **"cộng dồn + theo dõi max"** chỉ với hai biến, bộ nhớ `O(1)`.
4. Không quên **điểm xuất phát có độ cao 0** — cái bẫy chính của bài.
5. Biết dùng `itertools.accumulate` và hiểu nó làm gì bên dưới.
6. Tự giải các biến thể: độ cao thấp nhất, vị trí đạt đỉnh, đoạn leo dốc dài nhất…

## 1. Bài toán yêu cầu gì?

Một người đạp xe đi qua `n + 1` điểm, đánh số `0, 1, …, n`. Điểm `0` có độ cao **0**.

Mảng `gain` dài `n`: `gain[i]` là **độ chênh** khi đi từ điểm `i` sang điểm `i + 1` (dương = lên dốc, âm = xuống dốc).

Trả về **độ cao lớn nhất** trong tất cả các điểm (kể cả điểm 0).

| gain | Các độ cao | Kết quả |
|---|---|---:|
| `[-5,1,5,0,-7]` | `[0,-5,-4,1,1,-6]` | `1` |
| `[-4,-3,-2,-1,4,3,2]` | `[0,-4,-7,-9,-10,-6,-3,-1]` | `0` |

Constraints:

- `n == gain.length`, `1 <= n <= 100`
- `-100 <= gain[i] <= 100`

Chú ý ví dụ 2: mọi điểm sau xuất phát đều **âm**, nên đáp án là `0` — chính là điểm xuất phát.

### Vẽ hành trình để thấy rõ

Với `gain = [-5, 1, 5, 0, -7]`:

```text
điểm:     0     1     2     3     4     5
độ cao:   0 -> -5 -> -4 ->  1 ->  1 -> -6
gain:        -5    +1    +5    +0    -7
```

`gain` nằm **giữa** hai điểm, còn độ cao nằm **tại** điểm. Vì vậy có `n` giá trị gain nhưng `n + 1` độ cao.

In [1]:
gain = [-5, 1, 5, 0, -7]
print("số giá trị gain:", len(gain))
print("số điểm trên đường:", len(gain) + 1)

số giá trị gain: 5
số điểm trên đường: 6


## 2. Nền tảng Python cần biết

### 2.1. List và chỉ số

`gain[i]` đọc phần tử thứ `i` (đếm từ 0). `len(gain)` là số phần tử. `gain[-1]` là phần tử cuối.

In [2]:
gain = [-5, 1, 5, 0, -7]
print("gain[0] =", gain[0])
print("gain[2] =", gain[2])
print("gain[-1] =", gain[-1])
print("len =", len(gain))

gain[0] = -5
gain[2] = 5
gain[-1] = -7
len = 5


### 2.2. Vòng `for` duyệt trực tiếp phần tử

Khi chỉ cần **giá trị** (không cần chỉ số), viết `for g in gain:` gọn và ít lỗi hơn `for i in range(len(gain))`.

In [3]:
for g in gain:
    print("gain hiện tại:", g)

gain hiện tại: -5
gain hiện tại: 1
gain hiện tại: 5
gain hiện tại: 0
gain hiện tại: -7


### 2.3. Cộng dồn với `+=`

`altitude += g` nghĩa là `altitude = altitude + g`. Biến `altitude` "nhớ" tổng các giá trị đã cộng.

In [4]:
altitude = 0
for g in [-5, 1, 5]:
    altitude += g
    print("sau khi cộng", g, "-> altitude =", altitude)

sau khi cộng -5 -> altitude = -5
sau khi cộng 1 -> altitude = -4
sau khi cộng 5 -> altitude = 1


### 2.4. `append` để xây list mới

`lst.append(x)` thêm `x` vào **cuối** list. Ta sẽ dùng nó để lưu lại toàn bộ dãy độ cao.

In [5]:
altitudes = [0]
altitudes.append(-5)
altitudes.append(-4)
print(altitudes)

[0, -5, -4]


### 2.5. `max()` và hàm `max(a, b)`

- `max(lst)` trả phần tử lớn nhất của list.
- `max(a, b)` trả số lớn hơn trong hai số — rất tiện để cập nhật "kỷ lục".

In [6]:
print(max([0, -5, -4, 1, 1, -6]))
print(max(3, 7), max(-2, -9), max(0, 0))

1
7 -2 0


### 2.6. Tổng tiền tố (prefix sum) là gì?

Cho dãy `a`, **tổng tiền tố** `p[k]` = tổng `k` phần tử đầu: `p[0] = 0`, `p[k] = a[0] + … + a[k-1]`.

Ví dụ `a = [3, 1, 4]` → `p = [0, 3, 4, 8]`. Công thức truy hồi: `p[k] = p[k-1] + a[k-1]`.

Trong bài này: **độ cao tại điểm k chính là p[k]** của mảng `gain`. Đó là toàn bộ "bí mật" của bài.

In [7]:
a = [3, 1, 4]
p = [0]
for x in a:
    p.append(p[-1] + x)   # p[-1] là tổng tiền tố gần nhất
print("a =", a)
print("p =", p)

a = [3, 1, 4]
p = [0, 3, 4, 8]


## 3. Cách trực giác — tính hết các độ cao rồi lấy max

Người mới thường nghĩ: "Mình cứ đi theo người đạp xe, ghi lại độ cao từng điểm vào sổ, cuối cùng nhìn sổ tìm số lớn nhất."

Các bước:

1. `altitudes = [0]` (điểm xuất phát).
2. Với mỗi `g` trong `gain`: độ cao mới = độ cao cuối cùng + `g`, thêm vào sổ.
3. Trả `max(altitudes)`.

In [8]:
def largest_altitude_list(gain: list[int]) -> int:
    altitudes = [0]
    for g in gain:
        altitudes.append(altitudes[-1] + g)
    return max(altitudes)

print(largest_altitude_list([-5, 1, 5, 0, -7]))        # 1
print(largest_altitude_list([-4, -3, -2, -1, 4, 3, 2])) # 0

1
0


### 3.1. Một baseline "ngây thơ" hơn nữa: tính lại tổng từ đầu

Có người còn viết: độ cao điểm `k` = `sum(gain[:k])` — cắt `k` phần tử đầu rồi cộng.

`gain[:k]` là **slicing**: lấy từ chỉ số 0 tới trước `k`. Cách này đúng nhưng mỗi điểm lại cộng từ đầu → tổng cộng khoảng `n²/2` phép cộng.

In [9]:
gain = [-5, 1, 5, 0, -7]
for k in range(len(gain) + 1):
    print(f"k={k}: gain[:{k}] = {gain[:k]} -> sum = {sum(gain[:k])}")

k=0: gain[:0] = [] -> sum = 0
k=1: gain[:1] = [-5] -> sum = -5
k=2: gain[:2] = [-5, 1] -> sum = -4
k=3: gain[:3] = [-5, 1, 5] -> sum = 1
k=4: gain[:4] = [-5, 1, 5, 0] -> sum = 1
k=5: gain[:5] = [-5, 1, 5, 0, -7] -> sum = -6


In [10]:
def largest_altitude_naive(gain: list[int]) -> int:
    best = None
    for k in range(len(gain) + 1):
        h = sum(gain[:k])          # tính lại từ đầu: O(k)
        if best is None or h > best:
            best = h
    return best

print(largest_altitude_naive([-5, 1, 5, 0, -7]))
print(largest_altitude_naive([-4, -3, -2, -1, 4, 3, 2]))

1
0


## 4. Nút thắt và insight

**Nút thắt của cách ngây thơ:** `sum(gain[:k])` và `sum(gain[:k+1])` trùng nhau gần hết — ta cộng đi cộng lại cùng một phần.

**Insight 1 (prefix sum):** `độ cao[k+1] = độ cao[k] + gain[k]`. Chỉ cần nhớ độ cao hiện tại, cộng thêm một số là có điểm tiếp theo.

**Nút thắt của cách list:** ta lưu cả `n + 1` độ cao, nhưng đề **chỉ hỏi giá trị lớn nhất**. Ta không bao giờ cần nhìn lại độ cao cũ.

**Insight 2 (running max):** giữ biến `best` = độ cao cao nhất **đã gặp cho tới giờ**. Mỗi lần có độ cao mới, so với `best` rồi cập nhật. Không cần list → bộ nhớ `O(1)`.

**Cái bẫy:** `best` phải khởi tạo bằng **0** (độ cao điểm xuất phát), không phải `gain[0]` hay `-inf` rồi quên điểm 0.

In [11]:
# So sánh số phép cộng: ngây thơ vs cộng dồn
for n in [5, 10, 100]:
    naive_adds = sum(range(n + 1))   # 0 + 1 + ... + n
    running_adds = n
    print(f"n={n:>3}: ngây thơ ~{naive_adds:>5} phép cộng | cộng dồn {running_adds} phép cộng")

n=  5: ngây thơ ~   15 phép cộng | cộng dồn 5 phép cộng
n= 10: ngây thơ ~   55 phép cộng | cộng dồn 10 phép cộng
n=100: ngây thơ ~ 5050 phép cộng | cộng dồn 100 phép cộng


## 5. Lời giải chính — cộng dồn + theo dõi max

Ta giảng từng đoạn rồi ghép lại.

### 5.1. Hai biến trạng thái

- `altitude`: độ cao **tại điểm hiện tại**. Ban đầu `0`.
- `best`: độ cao **cao nhất đã gặp** (tính cả điểm hiện tại). Ban đầu `0` vì điểm 0 đã được "gặp".

In [12]:
altitude = 0
best = 0
print("trước vòng lặp: altitude =", altitude, "| best =", best)

trước vòng lặp: altitude = 0 | best = 0


### 5.2. Một bước di chuyển

Đi qua một đoạn có độ chênh `g`: cập nhật `altitude`, rồi so với `best`. Thứ tự quan trọng: **cộng trước, so sau** — vì ta so độ cao của điểm mới.

In [13]:
g = -5
altitude += g
best = max(best, altitude)
print("sau đoạn g=-5: altitude =", altitude, "| best =", best)

g = 1
altitude += g
best = max(best, altitude)
print("sau đoạn g=+1: altitude =", altitude, "| best =", best)

sau đoạn g=-5: altitude = -5 | best = 0
sau đoạn g=+1: altitude = -4 | best = 0


### 5.3. Ghép thành hàm hoàn chỉnh

In [14]:
def largest_altitude(gain: list[int]) -> int:
    altitude = 0          # độ cao tại điểm hiện tại (bắt đầu ở điểm 0)
    best = 0              # điểm 0 đã được tính
    for g in gain:
        altitude += g     # đi sang điểm kế tiếp
        best = max(best, altitude)
    return best

print(largest_altitude([-5, 1, 5, 0, -7]))         # 1
print(largest_altitude([-4, -3, -2, -1, 4, 3, 2]))  # 0
print(largest_altitude([52, -91, 72]))               # 52

1
0
52


## 6. TRACE từng bước

Bảng trace cho `gain = [-5, 1, 5, 0, -7]`:

| bước | g | altitude trước | altitude sau | best trước | so sánh | best sau |
|---:|---:|---:|---:|---:|---|---:|
| 1 | -5 | 0 | -5 | 0 | -5 > 0? Không | 0 |
| 2 | 1 | -5 | -4 | 0 | -4 > 0? Không | 0 |
| 3 | 5 | -4 | 1 | 0 | 1 > 0? **Có** | 1 |
| 4 | 0 | 1 | 1 | 1 | 1 > 1? Không | 1 |
| 5 | -7 | 1 | -6 | 1 | -6 > 1? Không | 1 |

Kết quả: `best = 1`. Hàm trace dưới đây in đúng các cột này.

In [15]:
def trace_largest_altitude(gain: list[int]) -> int:
    altitude, best = 0, 0
    print(f"gain = {gain}")
    print("bước |   g | alt trước -> alt sau | best trước | cập nhật? | best sau")
    for step, g in enumerate(gain, start=1):
        before = altitude
        altitude += g
        old_best = best
        updated = altitude > best
        best = max(best, altitude)
        print(f"{step:>4} | {g:>3} | {before:>9} -> {altitude:<7} | {old_best:>10} | {'CÓ' if updated else 'không':>9} | {best}")
    print("=> trả về", best)
    return best

trace_largest_altitude([-5, 1, 5, 0, -7])

gain = [-5, 1, 5, 0, -7]
bước |   g | alt trước -> alt sau | best trước | cập nhật? | best sau
   1 |  -5 |         0 -> -5      |          0 |     không | 0
   2 |   1 |        -5 -> -4      |          0 |     không | 0
   3 |   5 |        -4 -> 1       |          0 |        CÓ | 1
   4 |   0 |         1 -> 1       |          1 |     không | 1
   5 |  -7 |         1 -> -6      |          1 |     không | 1
=> trả về 1


1

### 6.1. Trace ví dụ "toàn âm" — thấy vai trò của điểm xuất phát

Ở ví dụ 2, `best` **không bao giờ được cập nhật**: nó giữ nguyên giá trị khởi tạo `0`. Nếu khởi tạo sai (ví dụ bằng `gain[0] = -4`), đáp án sẽ sai.

In [16]:
trace_largest_altitude([-4, -3, -2, -1, 4, 3, 2])

gain = [-4, -3, -2, -1, 4, 3, 2]
bước |   g | alt trước -> alt sau | best trước | cập nhật? | best sau
   1 |  -4 |         0 -> -4      |          0 |     không | 0
   2 |  -3 |        -4 -> -7      |          0 |     không | 0
   3 |  -2 |        -7 -> -9      |          0 |     không | 0
   4 |  -1 |        -9 -> -10     |          0 |     không | 0
   5 |   4 |       -10 -> -6      |          0 |     không | 0
   6 |   3 |        -6 -> -3      |          0 |     không | 0
   7 |   2 |        -3 -> -1      |          0 |     không | 0
=> trả về 0


0

### 6.2. `enumerate(..., start=1)` dùng trong trace

`enumerate(lst)` trả từng cặp `(chỉ_số, phần_tử)`. Tham số `start=1` cho chỉ số bắt đầu từ 1 — tiện để đánh số "bước 1, bước 2…".

In [17]:
for step, g in enumerate([-5, 1, 5], start=1):
    print("bước", step, "g =", g)

bước 1 g = -5
bước 2 g = 1
bước 3 g = 5


## 7. Code nộp LeetCode

Bản gọn, không in debug, đúng chữ ký `largestAltitude(self, gain)`.

In [18]:
class Solution:
    def largestAltitude(self, gain: list[int]) -> int:
        altitude = 0
        best = 0
        for g in gain:
            altitude += g
            if altitude > best:
                best = altitude
        return best

sol = Solution()
print(sol.largestAltitude([-5, 1, 5, 0, -7]))
print(sol.largestAltitude([-4, -3, -2, -1, 4, 3, 2]))

1
0


## 8. Các lời giải khác

### 8.1. `itertools.accumulate` — prefix sum dựng sẵn

`accumulate(iterable)` trả về các tổng cộng dồn: `accumulate([3,1,4])` → `3, 4, 8`.

Tham số `initial=0` (Python ≥ 3.8) chèn thêm `0` ở đầu → đúng là dãy độ cao có cả điểm xuất phát.

In [19]:
from itertools import accumulate

print(list(accumulate([3, 1, 4])))
print(list(accumulate([3, 1, 4], initial=0)))
print(list(accumulate([-5, 1, 5, 0, -7], initial=0)))

[3, 4, 8]
[0, 3, 4, 8]
[0, -5, -4, 1, 1, -6]


In [20]:
def largest_altitude_accumulate(gain: list[int]) -> int:
    return max(accumulate(gain, initial=0))

print(largest_altitude_accumulate([-5, 1, 5, 0, -7]))
print(largest_altitude_accumulate([-4, -3, -2, -1, 4, 3, 2]))

1
0


Code nộp LeetCode — cách `accumulate` (một dòng). `max` đọc trực tiếp từ iterator nên không tạo list, bộ nhớ vẫn `O(1)`.

In [21]:
from itertools import accumulate

class Solution:
    def largestAltitude(self, gain: list[int]) -> int:
        return max(accumulate(gain, initial=0))

print(Solution().largestAltitude([-5, 1, 5, 0, -7]))

1


### 8.2. Ghi đè ngay trên `gain` (in-place prefix sum)

Biến `gain` thành mảng prefix: `gain[i] += gain[i-1]`. Sau đó `gain[i]` = độ cao điểm `i+1`. Đáp án = `max(0, max(gain))`.

Bộ nhớ phụ `O(1)` nhưng **làm hỏng input** — trong thực tế nên tránh nếu người gọi còn dùng mảng.

In [22]:
def largest_altitude_inplace(gain: list[int]) -> int:
    gain = gain[:]                    # sao chép để demo không phá dữ liệu gốc
    for i in range(1, len(gain)):
        gain[i] += gain[i - 1]
    return max(0, max(gain))

print(largest_altitude_inplace([-5, 1, 5, 0, -7]))
print(largest_altitude_inplace([-4, -3, -2, -1, 4, 3, 2]))

1
0


### 8.3. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---:|---:|---|
| Ngây thơ `sum(gain[:k])` | Tính lại từ đầu mỗi điểm | `O(n²)` | `O(n)` (slice) | Chỉ để hiểu nút thắt |
| List độ cao | Lưu hết rồi `max` | `O(n)` | `O(n)` | Dễ debug, in được cả dãy |
| Cộng dồn + max | 2 biến | `O(n)` | `O(1)` | **Lời giải chính** |
| `accumulate(initial=0)` | Prefix dựng sẵn | `O(n)` | `O(1)` | Ngắn, cần Python ≥ 3.8 |
| In-place | Ghi đè `gain` | `O(n)` | `O(1)` | Phá input |

## 9. Vì sao lời giải đúng?

**Bất biến (invariant)** sau khi xử lý `k` phần tử đầu của `gain`:

- `altitude` = `gain[0] + … + gain[k-1]` = độ cao điểm `k`.
- `best` = `max(độ cao điểm 0, 1, …, k)`.

**Khởi đầu (k = 0):** `altitude = 0` = độ cao điểm 0; `best = 0` = max của tập `{0}`. Đúng.

**Duy trì:** đọc `gain[k]` → `altitude` mới = độ cao điểm `k+1`. `best = max(best cũ, altitude mới)` = max của điểm `0..k+1`. Đúng.

**Kết thúc (k = n):** `best` = max của mọi điểm `0..n` — đúng thứ đề hỏi. Không điểm nào bị bỏ sót vì mỗi điểm được so đúng một lần ngay khi tới.

In [23]:
# Kiểm chứng bất biến trên một ví dụ: sau mỗi bước, best == max(tiền tố độ cao)
gain = [3, -1, -4, 6, -2]
alt, best, heights = 0, 0, [0]
for g in gain:
    alt += g
    best = max(best, alt)
    heights.append(alt)
    assert alt == sum(gain[:len(heights) - 1])
    assert best == max(heights)
    print("độ cao đã đi:", heights, "| best =", best)
print("Bất biến đúng ở mọi bước")

độ cao đã đi: [0, 3] | best = 3
độ cao đã đi: [0, 3, 2] | best = 3
độ cao đã đi: [0, 3, 2, -2] | best = 3
độ cao đã đi: [0, 3, 2, -2, 4] | best = 4
độ cao đã đi: [0, 3, 2, -2, 4, 2] | best = 4
Bất biến đúng ở mọi bước


## 10. Độ phức tạp

Gọi `n = len(gain)`.

- **Time `O(n)`:** một vòng lặp, mỗi bước 1 phép cộng + 1 phép so sánh.
- **Space `O(1)`:** chỉ hai biến `altitude`, `best` (không tính input).

Không thể nhanh hơn `O(n)`: chỉ cần bỏ qua một `gain[i]` nào đó, nó có thể là `+100` và tạo ra đỉnh mới → phải đọc hết input.

Với `n <= 100`, mọi cách (kể cả `O(n²)`) đều qua dễ dàng; nhưng thói quen viết `O(n)` / `O(1)` mới là thứ cần luyện.

## 11. Bộ test

Nhóm test:

1. Hai ví dụ của đề.
2. Biên: `n = 1` (dương, âm, bằng 0).
3. Toàn âm → đáp án 0 (bẫy điểm xuất phát).
4. Đỉnh ở cuối, đỉnh ở giữa, có lúc quay lại đúng đỉnh cũ.
5. Đối chiếu tất cả các cách trên nhiều input ngẫu nhiên (có seed).

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS (={actual})")

sol = Solution()
check("đề - ví dụ 1", sol.largestAltitude([-5, 1, 5, 0, -7]), 1)
check("đề - ví dụ 2", sol.largestAltitude([-4, -3, -2, -1, 4, 3, 2]), 0)
check("n=1 dương", sol.largestAltitude([7]), 7)
check("n=1 âm", sol.largestAltitude([-7]), 0)
check("n=1 bằng 0", sol.largestAltitude([0]), 0)
check("toàn âm", sol.largestAltitude([-1, -1, -1]), 0)
check("đỉnh ở cuối", sol.largestAltitude([1, 2, 3]), 6)
check("đỉnh ở giữa", sol.largestAltitude([5, 5, -20, 3]), 10)
check("xuống rồi lên cao hơn", sol.largestAltitude([-10, 15]), 5)
check("max biên", sol.largestAltitude([100] * 100), 10000)
check("min biên", sol.largestAltitude([-100] * 100), 0)

đề - ví dụ 1                 PASS (=1)
đề - ví dụ 2                 PASS (=0)
n=1 dương                    PASS (=7)
n=1 âm                       PASS (=0)
n=1 bằng 0                   PASS (=0)
toàn âm                      PASS (=0)
đỉnh ở cuối                  PASS (=6)
đỉnh ở giữa                  PASS (=10)
xuống rồi lên cao hơn        PASS (=5)
max biên                     PASS (=10000)
min biên                     PASS (=0)


In [25]:
import random

rng = random.Random(1732)
methods = [largest_altitude_list, largest_altitude_naive, largest_altitude,
           largest_altitude_accumulate, largest_altitude_inplace,
           lambda g: Solution().largestAltitude(g)]
for _ in range(500):
    n = rng.randint(1, 30)
    gain = [rng.randint(-100, 100) for _ in range(n)]
    answers = {m(gain) for m in methods}
    assert len(answers) == 1, (gain, answers)
print("500 input ngẫu nhiên: mọi cách cho cùng kết quả")

500 input ngẫu nhiên: mọi cách cho cùng kết quả


## 12. Lỗi người mới hay gặp

### Lỗi 1: Khởi tạo `best = gain[0]` hoặc chỉ lấy max của prefix không có 0

Sai vì quên điểm xuất phát có độ cao 0.

In [26]:
def wrong_forget_start(gain):
    altitude = 0
    best = float("-inf")       # quên rằng điểm 0 có độ cao 0
    for g in gain:
        altitude += g
        best = max(best, altitude)
    return best

print("sai:", wrong_forget_start([-4, -3, -2, -1, 4, 3, 2]), "| đúng:", largest_altitude([-4, -3, -2, -1, 4, 3, 2]))

sai: -1 | đúng: 0


**Sửa:** khởi tạo `best = 0`, hoặc dùng `accumulate(..., initial=0)`.

### Lỗi 2: Lấy `max(gain)` — nhầm độ chênh với độ cao

`gain` là **độ chênh**, không phải độ cao. `max(gain)` chỉ cho đoạn dốc lên mạnh nhất.

In [27]:
gain = [-5, 1, 5, 0, -7]
print("max(gain) =", max(gain), "(sai) | đáp án đúng =", largest_altitude(gain))

max(gain) = 5 (sai) | đáp án đúng = 1


### Lỗi 3: So sánh trước khi cộng

Nếu viết `best = max(best, altitude)` **trước** `altitude += g`, điểm cuối cùng không bao giờ được so.

In [28]:
def wrong_order(gain):
    altitude, best = 0, 0
    for g in gain:
        best = max(best, altitude)   # so độ cao CŨ
        altitude += g
    return best                      # thiếu so điểm cuối

print("sai:", wrong_order([1, 2, 3]), "| đúng:", largest_altitude([1, 2, 3]))

sai: 3 | đúng: 6


**Sửa:** cộng trước, so sau (hoặc so thêm một lần sau vòng lặp).

### Lỗi 4: Viết `altitude = g` thay vì `altitude += g`

Mất tác dụng "cộng dồn" — `altitude` chỉ còn là gain hiện tại.

In [29]:
def wrong_assign(gain):
    altitude, best = 0, 0
    for g in gain:
        altitude = g          # quên dấu +
        best = max(best, altitude)
    return best

print("sai:", wrong_assign([2, 2, 2]), "| đúng:", largest_altitude([2, 2, 2]))

sai: 2 | đúng: 6


## 13. Mẹo nhận dạng pattern

- Đề cho **"độ chênh / thay đổi / net gain / delta"** giữa các bước và hỏi về **giá trị tại các bước** → nghĩ ngay **prefix sum**.
- Đề hỏi **max / min** của một dãy tính được tuần tự → **running max/min**, không cần lưu cả dãy.
- Luôn hỏi: "Giá trị khởi đầu là gì? Nó có được tính vào đáp án không?" (ở đây: có, độ cao 0).
- Từ khóa: *altitude, balance, net change, running total, cumulative*.

Bài cùng họ: 1480 Running Sum of 1d Array, 724 Find Pivot Index, 1413 Minimum Value to Get Positive Step by Step Sum, 303 Range Sum Query.

## 14. Biến thể

### 14.1. Độ cao **thấp nhất** (đổi `max` thành `min`)

In [30]:
def lowest_altitude(gain):
    altitude, low = 0, 0
    for g in gain:
        altitude += g
        low = min(low, altitude)
    return low

print(lowest_altitude([-5, 1, 5, 0, -7]))   # -6
print(lowest_altitude([3, 4]))              # 0

-6
0


### 14.2. Trả về **điểm** (chỉ số) đạt độ cao lớn nhất, lấy điểm sớm nhất

Thêm biến `best_idx`; chỉ cập nhật khi **lớn hơn hẳn** (`>`) để giữ chỉ số sớm nhất.

In [31]:
def peak_point(gain):
    altitude, best, best_idx = 0, 0, 0
    for i, g in enumerate(gain, start=1):   # i = chỉ số điểm vừa tới
        altitude += g
        if altitude > best:
            best, best_idx = altitude, i
    return best_idx

print(peak_point([-5, 1, 5, 0, -7]))  # 3 (điểm 3 và 4 đều cao 1, lấy 3)
print(peak_point([-1, -2]))           # 0

3
0


### 14.3. Liên hệ bài 1413: độ cao khởi đầu nhỏ nhất để không bao giờ < 1

Nếu độ cao thấp nhất (tính từ 0) là `low`, cần khởi đầu `1 - low` (và ít nhất 1).

In [32]:
def min_start_value(nums):
    return 1 - lowest_altitude(nums)

print(min_start_value([-3, 2, -3, 4, 2]))  # 5
print(min_start_value([1, 2]))             # 1

5
1


## 15. Bài tập tự luyện

Tự làm trước, **không** kéo xuống phần đáp án. Các hàm TODO không được gọi khi chạy notebook.

### Bài 1 — Tính tay

Với `gain = [2, -3, 4, -1, -5, 6]`, hãy viết dãy độ cao và cho biết đáp án. (Ghi ra giấy.)

### Bài 2 — Dự đoán output

`list(accumulate([1, -2, 3], initial=0))` in ra gì? `max(...)` của nó bằng bao nhiêu?

### Bài 3 — Điền TODO: dựng dãy độ cao

Viết `exercise_altitudes(gain)` trả về **list** `n + 1` độ cao (có điểm 0).

In [33]:
def exercise_altitudes(gain: list[int]) -> list[int]:
    # TODO 1: khởi tạo list với độ cao điểm 0
    # TODO 2: với mỗi g, thêm (độ cao cuối + g)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_altitudes([-5, 1, 5, 0, -7]) == [0, -5, -4, 1, 1, -6]

### Bài 4 — Tự viết lời giải O(1) bộ nhớ

Viết lại `largestAltitude` chỉ dùng hai biến, không nhìn code mẫu.

In [34]:
def exercise_largest_altitude(gain: list[int]) -> int:
    # TODO: altitude, best = ...; duyệt gain; cộng trước, so sau
    pass

# assert exercise_largest_altitude([-4, -3, -2, -1, 4, 3, 2]) == 0

### Bài 5 — Biến thể: chênh lệch độ cao lớn nhất giữa hai điểm bất kỳ

Trả về `max(độ cao) - min(độ cao)` trên toàn hành trình (có điểm 0), chỉ một vòng lặp.

In [35]:
def exercise_altitude_range(gain: list[int]) -> int:
    # TODO: theo dõi đồng thời max và min
    pass

# assert exercise_altitude_range([-5, 1, 5, 0, -7]) == 7

### Bài 6 — Mini challenge: quãng leo dốc liên tục lớn nhất

Tìm tổng độ cao **leo được nhiều nhất** trên một đoạn đường liên tiếp, tức `max(độ cao[j] - độ cao[i])` với `i <= j`. (Nếu chỉ xuống dốc, đáp án 0.)

Gợi ý: đây là bài "mua thấp bán cao" (LeetCode 121) trên dãy độ cao.

In [36]:
def exercise_best_climb(gain: list[int]) -> int:
    # TODO: duyệt độ cao, giữ độ cao thấp nhất đã gặp
    pass

# assert exercise_best_climb([-5, 1, 5, 0, -7]) == 6   # từ -5 lên 1
# assert exercise_best_climb([-1, -2]) == 0

## 16. Gợi ý 3 mức

**Bài 1** — M1: bắt đầu từ 0. M2: mỗi bước cộng một gain. M3: `0, 2, -1, 3, 2, -3, 3` → max?

**Bài 2** — M1: `initial=0` thêm 0 ở đầu. M2: cộng dồn 1, rồi -2, rồi 3. M3: `[0, 1, -1, 2]`.

**Bài 3** — M1: list bắt đầu `[0]`. M2: `heights[-1]` là phần tử cuối. M3: `heights.append(heights[-1] + g)`.

**Bài 4** — M1: hai biến đều bằng 0. M2: `altitude += g`. M3: `best = max(best, altitude)` sau khi cộng.

**Bài 5** — M1: max và min đều khởi tạo 0 (điểm xuất phát). M2: mỗi bước cập nhật cả hai. M3: trả `hi - lo`.

**Bài 6** — M1: muốn hiệu lớn nhất thì trừ đi "đáy" thấp nhất trước đó. M2: giữ `low` = min độ cao đã đi qua. M3: `best = max(best, alt - low)`, sau đó `low = min(low, alt)`.

## 17. Đáp án tham khảo

Chỉ xem sau khi đã tự làm.

**Bài 1:** độ cao `[0, 2, -1, 3, 2, -3, 3]` → đáp án `3`.

**Bài 2:** `[0, 1, -1, 2]`, max là `2`.

In [37]:
from itertools import accumulate
print(list(accumulate([1, -2, 3], initial=0)), max(accumulate([1, -2, 3], initial=0)))
print(largest_altitude([2, -3, 4, -1, -5, 6]))

[0, 1, -1, 2] 2
3


In [38]:
def answer_altitudes(gain):
    heights = [0]
    for g in gain:
        heights.append(heights[-1] + g)
    return heights


def answer_largest_altitude(gain):
    altitude, best = 0, 0
    for g in gain:
        altitude += g
        best = max(best, altitude)
    return best


def answer_altitude_range(gain):
    altitude, hi, lo = 0, 0, 0
    for g in gain:
        altitude += g
        hi = max(hi, altitude)
        lo = min(lo, altitude)
    return hi - lo


def answer_best_climb(gain):
    altitude, low, best = 0, 0, 0
    for g in gain:
        altitude += g
        best = max(best, altitude - low)
        low = min(low, altitude)
    return best


assert answer_altitudes([-5, 1, 5, 0, -7]) == [0, -5, -4, 1, 1, -6]
assert answer_largest_altitude([-4, -3, -2, -1, 4, 3, 2]) == 0
assert answer_largest_altitude([-5, 1, 5, 0, -7]) == 1
assert answer_altitude_range([-5, 1, 5, 0, -7]) == 7
assert answer_best_climb([-5, 1, 5, 0, -7]) == 6
assert answer_best_climb([-1, -2]) == 0

# đối chiếu best_climb với brute force O(n^2) trên dữ liệu ngẫu nhiên
rng = random.Random(17)
for _ in range(300):
    gain = [rng.randint(-10, 10) for _ in range(rng.randint(1, 12))]
    h = answer_altitudes(gain)
    brute = max(h[j] - h[i] for i in range(len(h)) for j in range(i, len(h)))
    assert answer_best_climb(gain) == brute
    assert answer_altitude_range(gain) == max(h) - min(h)
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def largestAltitude(self, gain: list[int]) -> int:
        altitude = best = 0          # điểm xuất phát cao 0
        for g in gain:
            altitude += g            # prefix sum
            best = max(best, altitude)  # running max
        return best
```

Template tổng quát "prefix + running max":

```text
state = giá trị khởi đầu   (đừng quên nó có thể là đáp án!)
best  = state
for delta in deltas:
    state += delta
    best = max(best, state)
```

Checklist trước khi submit:

- [ ] `best` khởi tạo bằng **0** (điểm xuất phát), không phải `gain[0]` hay `-inf`?
- [ ] Dùng `+=` (cộng dồn), không phải `=`?
- [ ] Cộng trước, so sau — điểm cuối cùng có được so không?
- [ ] Trả về **độ cao**, không phải chỉ số hay `max(gain)`?
- [ ] Đã thử ví dụ toàn âm (đáp án 0)?